# Постобробка U-Net: від маски до окремих кораблів

Етап 7. Беремо навчену U-Net (`05_unet_train.ipynb`), рахуємо ймовірності на val-знімках з кораблями
і перебираємо параметри постобробки (`src/postprocess.py`) за **метрикою змагання** (F2 по порогах IoU):
поріг ймовірності × мінімальна площа компоненти × зв'язність (4 або 8).

Локально ноутбук проходить на `data/train_micro/` з вагами `data/unet_v1/` (перевірка логіки),
на Kaggle — на всіх 4 256 val-знімках з кораблями.

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; секрет `KAGGLE_API_TOKEN` прикріплений.
2. Input: `airbus-ship-detection-src` останньої версії (з `postprocess.py`) і вихід ноутбука
   **`airbus-unet`** (Add Input → Your Work → Notebooks) — там ваги `unet_resnet34.pth`.
3. **Save Version → Save & Run All**. Результати — у вкладці **Output** збереженої версії.

## 0. Дані на Kaggle

Те саме, що в `05_unet_train.ipynb`: токен → бібліотеки → архів змагання → `train_v2/` і CSV.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4 segmentation-models-pytorch==0.5.0

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "train_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" train_ship_segmentations_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_files = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    print("знімків у train_v2:", n_files)
    assert n_files == 192556

## 1. Ймовірності U-Net на val

Поділ і val — ті самі, що під час навчання (`make_split`, seed 42, лише знімки з кораблями).
Ймовірності зберігаємо як `uint8` (p × 255): 4 256 × 384 × 384 байт ≈ 0.6 ГБ у пам'яті замість 2.5 ГБ у float32;
точності 1/255 для порогів 0.3–0.7 вистачає.

In [ ]:
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = KAGGLE_DATA_DIR
    IMAGES_DIR = DATA_DIR / "train_v2"
    OUT_DIR = Path("/kaggle/working")
    # Точний шлях до виходу іншого ноутбука залежить від Kaggle — шукаємо файл.
    WEIGHTS = next(Path("/kaggle/input").rglob("unet_resnet34.pth"))
    DEVICE = "cuda"
    NUM_WORKERS = 4
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    IMAGES_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "postprocess_local"
    WEIGHTS = DATA_DIR / "unet_v1" / "unet_resnet34.pth"
    DEVICE = "cpu"
    NUM_WORKERS = 0

from dataset import ShipSegmentationDataset, rles_by_image
from metric import IOU_THRESHOLDS, f2_image
from postprocess import mask_to_rles
from rle import IMAGE_SHAPE, rle_decode
from split import make_split

SEED = 42
BATCH_SIZE = 32
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Kaggle:", ON_KAGGLE, "| device:", DEVICE, "| ваги:", WEIGHTS)

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
rles = rles_by_image(masks)
if ON_KAGGLE:
    split = make_split(masks)
else:
    available = {p.name for p in IMAGES_DIR.iterdir()}
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)

val_df = split[(split["split"] == "val") & (split["n_ships"] > 0)].reset_index(drop=True)
val_dataset = ShipSegmentationDataset(val_df["ImageId"], rles, IMAGES_DIR, train=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"val: {len(val_df)} знімків з кораблями")

In [ ]:
model = smp.Unet(encoder_name="resnet34", encoder_weights=None, classes=1)
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model = model.to(DEVICE).eval()

start = time.time()
batches = []
with torch.no_grad():
    for images, _ in val_loader:
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images.to(DEVICE))
        probs_batch = torch.sigmoid(logits.float())[:, 0]
        batches.append((probs_batch * 255).round().to(torch.uint8).cpu().numpy())
probs = np.concatenate(batches)
print(f"ймовірності: {probs.shape}, {probs.nbytes / 1e9:.2f} ГБ, {time.time() - start:.0f} с")

# Для етапу 8 і аналізу локально; майже всі пікселі ≈ 0, тож стиснення дає дуже малий файл.
np.savez_compressed(OUT_DIR / "val_probs_384.npz", probs=probs, image_ids=val_df["ImageId"].values)

## 2. Перебір параметрів постобробки

Для кожного знімка ймовірності збільшуються до 768 **один раз**, потім для кожного порогу і зв'язності
шукаються компоненти. Фільтр площі нічого не перераховує — лише відкидає стовпці матриці IoU.

Щоб перебір ішов хвилини, а не години, IoU всіх пар «справжній корабель × компонента» рахуємо одним
`np.bincount`: кожному пікселю — пара міток (корабель i, компонента k), і кількість пікселів з парою (i, k) —
це їхній перетин. Тому F2 тут рахується з готової матриці IoU (`f2_from_ious`), а не через `f2_image`.
Спершу перевіряємо, що результат збігається з `f2_image`.

In [ ]:
def ship_labels(image_rles):
    # Кораблі в розмітці не перекриваються, тож усі вміщуються в одну карту міток.
    labels = np.zeros(IMAGE_SHAPE, dtype=np.int32)
    for i, rle in enumerate(image_rles, start=1):
        labels[rle_decode(rle) == 1] = i
    return labels, len(image_rles)


def ious_from_labels(true_labels, n_true, pred_labels, n_pred):
    pairs = true_labels.astype(np.int64) * (n_pred + 1) + pred_labels
    joint = np.bincount(pairs.ravel(), minlength=(n_true + 1) * (n_pred + 1)).reshape(n_true + 1, n_pred + 1)
    intersection = joint[1:, 1:]
    true_areas = joint[1:, :].sum(axis=1)
    pred_areas = joint[:, 1:].sum(axis=0)
    return intersection / (true_areas[:, None] + pred_areas[None, :] - intersection)


def f2_from_ious(ious):
    n_true, n_pred = ious.shape
    if n_true == 0 and n_pred == 0:
        return 1.0
    scores = []
    for threshold in IOU_THRESHOLDS:
        tp = (ious > threshold).any(axis=1).sum()
        scores.append(5 * tp / (5 * tp + 4 * (n_true - tp) + (n_pred - tp)))
    return np.mean(scores)


def upscale(probs_384):
    return cv2.resize(probs_384.astype(np.float32) / 255, IMAGE_SHAPE[::-1], interpolation=cv2.INTER_LINEAR)


for i in range(min(20, len(val_df))):
    image_id = val_df["ImageId"][i]
    true_labels, n_true = ship_labels(rles[image_id])
    mask = (upscale(probs[i]) > 0.5).astype(np.uint8)
    n_labels, pred_labels, _, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    fast = f2_from_ious(ious_from_labels(true_labels, n_true, pred_labels, n_labels - 1))
    true_masks = [rle_decode(rle) for rle in rles[image_id]]
    pred_masks = [rle_decode(rle) for rle in mask_to_rles(probs[i] / 255, threshold=0.5, connectivity=8)]
    slow = f2_image(true_masks, pred_masks)
    assert np.isclose(fast, slow), (image_id, fast, slow)
print("f2_from_ious збігається з f2_image на", min(20, len(val_df)), "знімках: OK")

In [ ]:
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
CONNECTIVITIES = [4, 8]
MIN_AREAS = [0, 10, 20, 40, 80, 150]

start = time.time()
rows = []
for i, image_id in enumerate(val_df["ImageId"]):
    true_labels, n_true = ship_labels(rles[image_id])
    probs_768 = upscale(probs[i])
    for threshold in THRESHOLDS:
        mask = (probs_768 > threshold).astype(np.uint8)
        for connectivity in CONNECTIVITIES:
            n_labels, pred_labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=connectivity)
            ious = ious_from_labels(true_labels, n_true, pred_labels, n_labels - 1)
            areas = stats[1:, cv2.CC_STAT_AREA]
            for min_area in MIN_AREAS:
                rows.append((image_id, threshold, connectivity, min_area, f2_from_ious(ious[:, areas >= min_area])))
    if (i + 1) % 1000 == 0:
        print(f"{i + 1} знімків, {time.time() - start:.0f} с")

scores = pd.DataFrame(rows, columns=["ImageId", "threshold", "connectivity", "min_area", "f2"])
grid = scores.groupby(["threshold", "connectivity", "min_area"])["f2"].mean().reset_index()
grid.to_csv(OUT_DIR / "postprocess_grid.csv", index=False)
print(f"перебір: {len(grid)} комбінацій, {time.time() - start:.0f} с")

### 2.1. Результати

F2 на знімках з кораблями (середнє по знімках) для кожної комбінації. Для порівняння — стеля з
oracle (`src/postprocess.py`, правильні маски): **0.743** для 384 з 8-зв'язністю. Скор на всій val залежить
ще й від класифікатора (етап 8).

In [ ]:
for connectivity in CONNECTIVITIES:
    table = grid[grid["connectivity"] == connectivity].pivot(index="threshold", columns="min_area", values="f2")
    print(f"\n{connectivity}-зв'язність (рядки — поріг ймовірності, стовпці — min_area у пікселях 768):")
    print(table.round(4).to_string())

best = grid.loc[grid["f2"].idxmax()]
print(f"\nнайкраще: поріг {best['threshold']}, {int(best['connectivity'])}-зв'язність, "
      f"min_area {int(best['min_area'])} → F2 {best['f2']:.4f}")

In [ ]:
best_scores = scores[
    (scores["threshold"] == best["threshold"])
    & (scores["connectivity"] == best["connectivity"])
    & (scores["min_area"] == best["min_area"])
].reset_index(drop=True)
best_scores = best_scores.merge(val_df[["ImageId", "n_ships"]], on="ImageId")
best_scores[["ImageId", "n_ships", "f2"]].to_csv(OUT_DIR / "val_f2.csv", index=False)

print("F2 за кількістю кораблів на знімку:")
groups = pd.cut(best_scores["n_ships"], [0, 1, 2, 4, 99], labels=["1", "2", "3–4", "5+"])
print(best_scores.groupby(groups, observed=True)["f2"].agg(["mean", "count"]).round(3).to_string())
print(f"\nF2 = 0: {(best_scores['f2'] == 0).mean():.1%} знімків, F2 = 1: {(best_scores['f2'] == 1).mean():.1%}")

### 2.2. Приклади

4 випадкові val-знімки в повній роздільності: справжні кораблі і знайдені компоненти з найкращими параметрами.
Кожен корабель / компонента — свій колір: так видно злиття (одна компонента на два кораблі) і розпад.

In [ ]:
rng = np.random.default_rng(SEED)
examples = rng.choice(len(val_df), size=min(4, len(val_df)), replace=False)

fig, axes = plt.subplots(2, len(examples), figsize=(5 * len(examples), 10), squeeze=False)
for col, i in enumerate(examples):
    image_id = val_df["ImageId"][i]
    image = cv2.cvtColor(cv2.imread(str(IMAGES_DIR / image_id)), cv2.COLOR_BGR2RGB)
    true_labels, _ = ship_labels(rles[image_id])
    mask = (upscale(probs[i]) > best["threshold"]).astype(np.uint8)
    n_labels, pred_labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=int(best["connectivity"]))
    small = np.where(stats[:, cv2.CC_STAT_AREA] < best["min_area"])[0]
    pred_labels[np.isin(pred_labels, small[small > 0])] = 0
    f2 = best_scores.loc[best_scores["ImageId"] == image_id, "f2"].item()
    for row, (labels, title) in enumerate([(true_labels, "правда"), (pred_labels, f"передбачення, F2 {f2:.2f}")]):
        axes[row, col].imshow(image)
        axes[row, col].imshow(np.ma.masked_equal(labels, 0), cmap="tab10", alpha=0.6, interpolation="nearest")
        axes[row, col].set_title(f"{title}: {image_id}, об'єктів {len(np.unique(labels)) - 1}")
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "postprocess_examples.png", dpi=80)
plt.show()